# V025: calibrate the pruning rule from actual completed validation logs

Use three directly comparable unaugmented deep curve-loss runs: the original model and completed span3/span7 hybrids. They share the exact fixed A arrays, dataset size, optimizer steps, rate schedule, objective and numeric runtime. All are independently audited. The augmented mixup run belongs to a different fitting regime and is excluded from this cohort. No training-MAE or leaderboard values enter calibration.

The conservative policy is frozen before new training outcomes. A retrospective safety replay removes each target run from its reference envelope; this describes observed histories rather than guaranteeing future convergence.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,gc
import torch,numpy as np

def run_pruning_calibration_v025(root='.'):
    P=Path(root).resolve();O=P/'outputs/pruning_workflow_v025';O.mkdir(exist_ok=True)
    target=P/'research/pruning_policy_v025.json'
    if target.exists():raise FileExistsError('Preserve frozen policy')
    def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
    def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
    component=P/'research/pruning_components_v025.ipynb';component_code=code(component)
    ns={};exec(compile(component_code,str(component),'exec'),ns)
    specs=[('v013_plain_deep_curve','plain_deep_curve','outputs/controlled_experiments_v013/deep_curve','research/screen_review_v013.json'),
        ('v024_conv3','conv3','outputs/controlled_experiments_v024/conv3_curve','research/screen_review_v024.json'),
        ('v024_conv7','conv7','outputs/controlled_experiments_v024/conv7_curve','research/screen_review_v024.json')]
    references=[];variants={};cohort=None;required={}
    for run_id,variant,path,audit_path in specs:
        stage=P/path;report=json.loads((stage/'report.json').read_text());record=json.loads((stage/'record.json').read_text())
        audit=json.loads((P/audit_path).read_text())
        assert audit['status']=='passed' and report['completed'] and report['executed_epochs']==200
        assert sha(stage/'model.pt')==record['checkpoint_sha256'] and sha(stage/'report.json')==record['report_sha256']
        saved=torch.load(stage/'model.pt',map_location='cpu',weights_only=False);fp=saved['fingerprint']
        assert fp['mode']=='validation' and 'v021_mixup' not in fp
        arrays={k:fp[k] for k in ['x_sha256','y_sha256','out_sha256']}
        arrays.update({k:fp['extra'][k] for k in ['val_x_sha256','val_y_sha256','val_out_sha256']})
        c=dict(array_fingerprints=arrays,model_spec=report['model_spec'],feature_names=report['preprocessing']['feature_names'],
            fit_breaths=report['training_breaths'],validation_breaths=report['validation_breaths'],
            scored_validation_points=report['scored_validation_points'],train_cfg=report['train_cfg'],
            budget_epochs=report['executed_epochs'],planned_LR_schedule=report['executed_lr_schedule'],
            objective=dict(loss_kind=report['v008']['loss_kind'],curve_aux_weight=report['v008']['curve_aux_weight']),
            numeric_runtime=fp['numeric_runtime'])
        if cohort is None:cohort=c
        else:assert c==cohort,'Unmatched cohort'
        history=report['epochs']
        references.append(dict(id=run_id,variant=variant,seed=report['seed'],report_file=path+'/report.json',
            report_sha256=sha(stage/'report.json'),checkpoint_sha256=record['checkpoint_sha256'],audit_file=audit_path,
            audit_sha256=sha(P/audit_path),validation_inspiratory_MAE=[r['validation_inspiratory_mae'] for r in history],
            best_validation_inspiratory_MAE=[r['best_val_mae'] for r in history],
            optimizer_steps=[r['cumulative_optimizer_steps'] for r in history],LR=[r['lr_used'] for r in history]))
        names=['feature_components_v005.ipynb','sequence_components_v005.ipynb','training_components_v008.ipynb']
        if variant.startswith('conv'):names.append('convolution_components_v024.ipynb')
        hashes={k:report['v008']['source_fingerprints'][k] for k in names}
        codes=[report['v008']['factory_identity']['code_sha256']]
        if variant=='plain_deep_curve':codes.append(None) # Native V5 builder has the same pinned implementation.
        variants[variant]=dict(allowed_factory_code_sha256=codes,model_source_hashes=hashes,
            representative_ids=[run_id],description='Known completed representation; new variants finish a representative.')
        for file in [stage/'report.json',stage/'record.json',P/audit_path]:required[str(file.relative_to(P))]=sha(file)
        del saved;gc.collect()
    policy=ns['calibrate_pruning_v025'](cohort,references,variants)
    policy.update(created_utc=datetime.now(timezone.utc).isoformat(),origin='reviewed_completed_A_validation_curves',
        component_code_sha256=hashlib.sha256(component_code.encode()).hexdigest(),
        calibration_code_sha256=hashlib.sha256(code(P/'research/pruning_calibration_v025.ipynb').encode()).hexdigest(),
        reference_file_hashes=required,
        exclusions=[dict(run='V021 matched mixup',reason='Augmented fitting inputs/targets change convergence regime; excluded from the unaugmented comparison cohort.'),
            dict(run='Other splits/data sizes/optimizers/schedules or unfamiliar architectures',reason='Require compatible completed representatives before pruning.')],
        draft_revision='An initial raw-spike/max-duration-times2 draft was rejected before deployment: it offered no eligible stopping window under this cosine schedule. Final margin uses the same prefix-best statistic as the decision; patience and cooldown cover observed maximum recovery intervals.')
    target.write_text(json.dumps(policy,indent=2,allow_nan=False)+'\n')
    # Leave each target curve out of its comparison envelope; retain frozen calibration constants.
    replay=[]
    for ref in references:
        variant=ref['variant'];descriptor=dict(cohort=cohort,variant=variant,run_id=ref['id'],
            factory_identity=dict(code_sha256=variants[variant]['allowed_factory_code_sha256'][0]),
            model_source_hashes=variants[variant]['model_source_hashes'])
        history=[];stop=None
        for j in range(cohort['budget_epochs']):
            history.append(dict(epoch=j+1,cumulative_optimizer_steps=ref['optimizer_steps'][j],
                best_val_mae=ref['best_validation_inspiratory_MAE'][j],lr_used=ref['LR'][j]))
            d=ns['evaluate_pruning_v025'](history,descriptor,policy)
            if d['action']=='prune':stop=d;break
        replay.append(dict(target=ref['id'],target_reference_excluded=True,pruned=stop is not None,decision=stop))
    report=dict(status='calibrated_and_retrospectively_checked',created_utc=datetime.now(timezone.utc).isoformat(),
        policy_file=str(target.relative_to(P)),policy_sha256=sha(target),reference_count=len(references),
        relative_deficit_margin=policy['relative_deficit_margin'],stall_relative_tolerance=policy['stall_relative_tolerance'],
        patience_intervals=policy['patience_intervals'],LR_cooldown_evaluations=policy['LR_cooldown_evaluations'],
        start_comparing_epoch=policy['start_comparing_epoch'],budget_epochs=cohort['budget_epochs'],
        retrospectively_pruned_runs=sum(x['pruned'] for x in replay),leave_target_out_replays=replay,
        training_MAE_used=False,leaderboard_scores_used=False,approved_data_fits=0,
        interpretation='Retrospective engineering safety check on limited observed histories; not a statistical guarantee for unseen runs.')
    assert report['retrospectively_pruned_runs']==0,'Conservative policy unexpectedly pruned an observed representative'
    (P/'research/pruning_calibration_review_v025.json').write_text(json.dumps(report,indent=2)+'\n')
    print(json.dumps(report,indent=2))
    return policy,report

print('Inert until run_pruning_calibration_v025(project_root) is called.')


Inert until run_pruning_calibration_v025(project_root) is called.
{
  "status": "calibrated_and_retrospectively_checked",
  "created_utc": "2026-11-08T03:47:15.572168+00:00",
  "policy_file": "research/pruning_policy_v025.json",
  "policy_sha256": "0acdfa933d54fbae1e7728e2f8982bdae3f4227c6bd5187979bcd38b67e5fc88",
  "reference_count": 3,
  "relative_deficit_margin": 0.07084486505182362,
  "stall_relative_tolerance": 0.010095403951434384,
  "patience_intervals": 20,
  "LR_cooldown_evaluations": 13,
  "start_comparing_epoch": 100,
  "budget_epochs": 200,
  "retrospectively_pruned_runs": 0,
  "leave_target_out_replays": [
    {
      "target": "v013_plain_deep_curve",
      "target_reference_excluded": true,
      "pruned": false,
      "decision": null
    },
    {
      "target": "v024_conv3",
      "target_reference_excluded": true,
      "pruned": false,
      "decision": null
    },
    {
      "target": "v024_conv7",
      "target_reference_excluded": true,
      "pruned": false,
  